In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def make_preprocessor(scale=True):
    # Imputer, scaler and encoder are fitted inside each fold, never on the validation rows
    num_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale:  # needed for Ridge/Lasso/KNN, not for trees
        num_steps.append(('scale', StandardScaler()))
    cat_steps = [('impute', SimpleImputer(strategy='most_frequent')),
                 ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))]
    return ColumnTransformer([
        ('num', Pipeline(num_steps), NUMERIC_COLS),
        ('cat', Pipeline(cat_steps), CATEGORICAL_COLS),
    ])


def run_model(name, family, model, param_grid=None, scale=True):
    pipe = Pipeline([('prep', make_preprocessor(scale)), ('model', model)])
    inner_cv = KFold(n_splits=INNER_FOLDS, shuffle=True, random_state=SEED)
    best_params = []

    def fit_and_tune(X_tr, y_tr):
        if not param_grid:  # no hyperparameters (baseline, OLS)
            return pipe.fit(X_tr, y_tr), {}
        search = GridSearchCV(pipe, param_grid, cv=inner_cv,
                              scoring='neg_root_mean_squared_error', n_jobs=-1)
        search.fit(X_tr, y_tr)  # refit=True, so best_estimator_ is trained on all of X_tr
        return search.best_estimator_, search.best_params_

    # Outer loop: tune on the outer-train rows, score on the outer-validation rows
    for fold, (tr_idx, va_idx) in enumerate(k_folds):
        X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        y_tr, y_va = Y_train.iloc[tr_idx], Y_train.iloc[va_idx]
        fitted, params = fit_and_tune(X_tr, y_tr)
        best_params.append(params)
        record(cv_results, name, family, fold, y_va, fitted.predict(X_va))

    # Final fit on all training rows, then the test set is scored once
    fitted, params = fit_and_tune(X_train, Y_train)
    record(test_results, name, family, None, Y_test, fitted.predict(X_test))
    print(f'{name}: done. Final hyperparameters: {params}')
    return fitted, best_params

import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge

# Baseline and plain OLS have nothing to tune
run_model('Mean only', 'baseline', DummyRegressor(strategy='mean'))
run_model('OLS', 'linear', LinearRegression())

# Ridge: tune the penalty on the inner folds (alpha grid is on a log scale)
ridge_fit, ridge_best = run_model('Ridge', 'linear', Ridge(),
                                  param_grid={'model__alpha': np.logspace(-2, 3, 20)})

In [ ]:
import pandas as pd

cv_df = pd.DataFrame(cv_results)
test_df = pd.DataFrame(test_results).set_index('model')

# Outer-fold RMSE: mean and spread over the 20 outer folds, next to the single test score
summary = cv_df.groupby('model')['rmse'].agg(['mean', 'std']).round(3)
summary['test_rmse'] = test_df['rmse'].round(3)
print(summary.sort_values('mean'))

# Ridge's chosen alpha should be stable across folds; wildly different values would be a warning sign
print(pd.Series([p['model__alpha'] for p in ridge_best]).describe().round(3))

## Main Script: cross-validation and tuning

All the tuning happens on the 85% training set. The 15% test set is not used until the end.

1. `k_folds` (built once above) holds 20 train/validation splits of the training set. Every model uses the same 20 splits, so the scores can be compared fold by fold.
2. For each model, one `GridSearchCV` tries every hyperparameter combination on these 20 splits. The whole pipeline (imputer, scaler, encoder, model) is refit inside each split, so the validation rows never leak into preprocessing.
3. The combination with the lowest mean validation RMSE is the best one. Its 20 fold scores go into `cv_results`, and the model is refit on the full training set.
4. We compare the models using `cv_results`, pick a champion, and score it on the test set once.

One thing to remember: the best CV score is a little optimistic because we picked the best of several settings using the same folds. That is why the test set is still needed.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso

searches = {}   # the fitted GridSearchCV of each model

# name: (model, grid, scale?). Trees don't need scaling. Add the other models here later
MODEL_SPECS = {
    'Mean only': (DummyRegressor(strategy='mean'), None, True),
    'OLS':       (LinearRegression(), None, True),
    'Ridge':     (Ridge(), ridge_grid, True),
    'Lasso':     (Lasso(max_iter=20000), lasso_grid, True),
    # 'KNN':     (KNeighborsRegressor(), knn_grid, True),
    # 'Tree':    (DecisionTreeRegressor(random_state=SEED), tree_grid, False),
    # 'Forest':  (RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1), rf_grid, False),
}

for name, (model, grid, scale) in MODEL_SPECS.items():
    tune_model(name, model, grid, scale)

In [ ]:

cv_results = []    # validation scores on the outer folds
test_results = []  # final test-set scores, one per model

def record(results, model_name, y_true, y_pred):
    results.append({
        'model': model_name,
        'rmse': root_mean_squared_error(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
    })

In [ ]:
import pandas as pd

cv_df = pd.DataFrame(cv_results)
summary = cv_df.groupby('model')[['rmse', 'mae', 'r2']].agg(['mean', 'std']).round(3)
print(summary.sort_values(('rmse', 'mean')))

# best hyperparameters of each model
for name, search in searches.items():
    print(name, search.best_params_)